# Isolated Model 1 Colab benchmark (M1-COLAB-20261006)
Free compute only; one account and runtime. 40 epochs, seed 0, frozen source e0424f99e3f668e1722e56fac4f17372cd60a62d.
12 cumulative GPU allocation hours within 72 calendar hours from first allocation, including setup, diagnostics, staging and recovery.
At most two connection attempts per UTC day; no retry loops. Keep this notebook private.
Storage: one Drive folder through a drive.file-scoped rclone credential (Colab secret `TRUESKATE_BENCH_RCLONE`). It can reach only files its own OAuth client created; never mount Drive.
Use CPU for preparation and ledger cells. Each GPU cell releases its runtime after the phase, failure or budget expiry.
No test, training-set or new-park holdout evaluation. Sequential decoding, full precision and content verification; caching disabled throughout Colab.
Do not Run All. Execute the numbered steps individually.

## 1. CPU: verify the scoped inputs by Drive checksums and record the first connection attempt

In [ ]:
# Bootstrap: pinned rclone, drive.file-scoped credential, harness verified against this notebook.
import hashlib, importlib.util, io, os, subprocess, time, urllib.request, zipfile
from pathlib import Path
from google.colab import userdata
BIN, CONF, STORE = Path('/content/rclone/rclone'), Path('/content/rclone/rclone.conf'), Path('/content/bench-store')
EXPECTED = {'code/benchmark.py': 'fd8d34a6496ffa14ab58565e372c8a7f8dd350042a419b263685cf2149494ac2', 'code/supervisor.py': '81fbaf4df576dc8fc3046dcc3ad9dfd1b31802cee425879b35dc592542509c0b', 'inputs/input-lock.json': '0ceb4b72fd40fe810c77eaa517e52bf603254404bcb6be5c468e14b691abcc9b'}
if not BIN.exists():
    archive = urllib.request.urlopen('https://downloads.rclone.org/v1.75.1/rclone-v1.75.1-linux-amd64.zip', timeout=300).read()
    if hashlib.sha256(archive).hexdigest() != '982b5aa772841168f8e380f139e9e787b2a105403e32b94da8676a0e1c0a13ab':
        raise RuntimeError('rclone archive checksum mismatch')
    BIN.parent.mkdir(parents=True, exist_ok=True)
    BIN.write_bytes(zipfile.ZipFile(io.BytesIO(archive)).read('rclone-v1.75.1-linux-amd64/rclone')); BIN.chmod(0o700)
conf = userdata.get('TRUESKATE_BENCH_RCLONE')
if 'scope = drive.file' not in conf.splitlines() or not any(l.startswith('root_folder_id = ') for l in conf.splitlines()):
    raise RuntimeError('refusing a credential broader than one drive.file folder')
CONF.parent.mkdir(parents=True, exist_ok=True); CONF.write_text(conf); CONF.chmod(0o600)
os.environ.update(RCLONE_CONFIG=str(CONF), RCLONE_BIN=str(BIN))
for rel, digest in EXPECTED.items():
    subprocess.run([str(BIN), 'copyto', 'bench:' + rel, str(STORE / rel)], check=True)
    if hashlib.sha256((STORE / rel).read_bytes()).hexdigest() != digest:
        raise RuntimeError(rel + ' does not match this notebook')
spec = importlib.util.spec_from_file_location('supervisor', STORE / 'code/supervisor.py')
s = importlib.util.module_from_spec(spec); spec.loader.exec_module(s)
s.prepare()


## 2. Select free T4 (standard RAM), then run preflight and epochs 1–5. The 40-epoch cosine schedule is unchanged.

In [ ]:
# Bootstrap: pinned rclone, drive.file-scoped credential, harness verified against this notebook.
import hashlib, importlib.util, io, os, subprocess, time, urllib.request, zipfile
from pathlib import Path
from google.colab import userdata
BIN, CONF, STORE = Path('/content/rclone/rclone'), Path('/content/rclone/rclone.conf'), Path('/content/bench-store')
EXPECTED = {'code/benchmark.py': 'fd8d34a6496ffa14ab58565e372c8a7f8dd350042a419b263685cf2149494ac2', 'code/supervisor.py': '81fbaf4df576dc8fc3046dcc3ad9dfd1b31802cee425879b35dc592542509c0b', 'inputs/input-lock.json': '0ceb4b72fd40fe810c77eaa517e52bf603254404bcb6be5c468e14b691abcc9b'}
if not BIN.exists():
    archive = urllib.request.urlopen('https://downloads.rclone.org/v1.75.1/rclone-v1.75.1-linux-amd64.zip', timeout=300).read()
    if hashlib.sha256(archive).hexdigest() != '982b5aa772841168f8e380f139e9e787b2a105403e32b94da8676a0e1c0a13ab':
        raise RuntimeError('rclone archive checksum mismatch')
    BIN.parent.mkdir(parents=True, exist_ok=True)
    BIN.write_bytes(zipfile.ZipFile(io.BytesIO(archive)).read('rclone-v1.75.1-linux-amd64/rclone')); BIN.chmod(0o700)
conf = userdata.get('TRUESKATE_BENCH_RCLONE')
if 'scope = drive.file' not in conf.splitlines() or not any(l.startswith('root_folder_id = ') for l in conf.splitlines()):
    raise RuntimeError('refusing a credential broader than one drive.file folder')
CONF.parent.mkdir(parents=True, exist_ok=True); CONF.write_text(conf); CONF.chmod(0o600)
os.environ.update(RCLONE_CONFIG=str(CONF), RCLONE_BIN=str(BIN))
for rel, digest in EXPECTED.items():
    subprocess.run([str(BIN), 'copyto', 'bench:' + rel, str(STORE / rel)], check=True)
    if hashlib.sha256((STORE / rel).read_bytes()).hexdigest() != digest:
        raise RuntimeError(rel + ' does not match this notebook')
spec = importlib.util.spec_from_file_location('supervisor', STORE / 'code/supervisor.py')
s = importlib.util.module_from_spec(spec); spec.loader.exec_module(s)
s.execute('timing')


## 3. After deletion is observed in Runtime → Manage sessions: select CPU, close that allocation and record the next attempt. After an interruption, set SESSION_ID to the dropped session and rerun the matching GPU step.

In [ ]:
# Bootstrap: pinned rclone, drive.file-scoped credential, harness verified against this notebook.
import hashlib, importlib.util, io, os, subprocess, time, urllib.request, zipfile
from pathlib import Path
from google.colab import userdata
BIN, CONF, STORE = Path('/content/rclone/rclone'), Path('/content/rclone/rclone.conf'), Path('/content/bench-store')
EXPECTED = {'code/benchmark.py': 'fd8d34a6496ffa14ab58565e372c8a7f8dd350042a419b263685cf2149494ac2', 'code/supervisor.py': '81fbaf4df576dc8fc3046dcc3ad9dfd1b31802cee425879b35dc592542509c0b', 'inputs/input-lock.json': '0ceb4b72fd40fe810c77eaa517e52bf603254404bcb6be5c468e14b691abcc9b'}
if not BIN.exists():
    archive = urllib.request.urlopen('https://downloads.rclone.org/v1.75.1/rclone-v1.75.1-linux-amd64.zip', timeout=300).read()
    if hashlib.sha256(archive).hexdigest() != '982b5aa772841168f8e380f139e9e787b2a105403e32b94da8676a0e1c0a13ab':
        raise RuntimeError('rclone archive checksum mismatch')
    BIN.parent.mkdir(parents=True, exist_ok=True)
    BIN.write_bytes(zipfile.ZipFile(io.BytesIO(archive)).read('rclone-v1.75.1-linux-amd64/rclone')); BIN.chmod(0o700)
conf = userdata.get('TRUESKATE_BENCH_RCLONE')
if 'scope = drive.file' not in conf.splitlines() or not any(l.startswith('root_folder_id = ') for l in conf.splitlines()):
    raise RuntimeError('refusing a credential broader than one drive.file folder')
CONF.parent.mkdir(parents=True, exist_ok=True); CONF.write_text(conf); CONF.chmod(0o600)
os.environ.update(RCLONE_CONFIG=str(CONF), RCLONE_BIN=str(BIN))
for rel, digest in EXPECTED.items():
    subprocess.run([str(BIN), 'copyto', 'bench:' + rel, str(STORE / rel)], check=True)
    if hashlib.sha256((STORE / rel).read_bytes()).hexdigest() != digest:
        raise RuntimeError(rel + ' does not match this notebook')
spec = importlib.util.spec_from_file_location('supervisor', STORE / 'code/supervisor.py')
s = importlib.util.module_from_spec(spec); spec.loader.exec_module(s)
SESSION_ID = 'session-1'
s.confirm_release(SESSION_ID, time.time())
s.next_attempt()


## 4. Select free T4 again; resume at epoch 6 toward epoch 40. The harness checks restored model/optimizer/RNG/shuffle state and the cosine learning rate.

In [ ]:
# Bootstrap: pinned rclone, drive.file-scoped credential, harness verified against this notebook.
import hashlib, importlib.util, io, os, subprocess, time, urllib.request, zipfile
from pathlib import Path
from google.colab import userdata
BIN, CONF, STORE = Path('/content/rclone/rclone'), Path('/content/rclone/rclone.conf'), Path('/content/bench-store')
EXPECTED = {'code/benchmark.py': 'fd8d34a6496ffa14ab58565e372c8a7f8dd350042a419b263685cf2149494ac2', 'code/supervisor.py': '81fbaf4df576dc8fc3046dcc3ad9dfd1b31802cee425879b35dc592542509c0b', 'inputs/input-lock.json': '0ceb4b72fd40fe810c77eaa517e52bf603254404bcb6be5c468e14b691abcc9b'}
if not BIN.exists():
    archive = urllib.request.urlopen('https://downloads.rclone.org/v1.75.1/rclone-v1.75.1-linux-amd64.zip', timeout=300).read()
    if hashlib.sha256(archive).hexdigest() != '982b5aa772841168f8e380f139e9e787b2a105403e32b94da8676a0e1c0a13ab':
        raise RuntimeError('rclone archive checksum mismatch')
    BIN.parent.mkdir(parents=True, exist_ok=True)
    BIN.write_bytes(zipfile.ZipFile(io.BytesIO(archive)).read('rclone-v1.75.1-linux-amd64/rclone')); BIN.chmod(0o700)
conf = userdata.get('TRUESKATE_BENCH_RCLONE')
if 'scope = drive.file' not in conf.splitlines() or not any(l.startswith('root_folder_id = ') for l in conf.splitlines()):
    raise RuntimeError('refusing a credential broader than one drive.file folder')
CONF.parent.mkdir(parents=True, exist_ok=True); CONF.write_text(conf); CONF.chmod(0o600)
os.environ.update(RCLONE_CONFIG=str(CONF), RCLONE_BIN=str(BIN))
for rel, digest in EXPECTED.items():
    subprocess.run([str(BIN), 'copyto', 'bench:' + rel, str(STORE / rel)], check=True)
    if hashlib.sha256((STORE / rel).read_bytes()).hexdigest() != digest:
        raise RuntimeError(rel + ' does not match this notebook')
spec = importlib.util.spec_from_file_location('supervisor', STORE / 'code/supervisor.py')
s = importlib.util.module_from_spec(spec); spec.loader.exec_module(s)
s.execute('completion')


## 5. After final deletion is observed: on CPU, close the last allocation and write the report. Record no further attempt.

In [ ]:
# Bootstrap: pinned rclone, drive.file-scoped credential, harness verified against this notebook.
import hashlib, importlib.util, io, os, subprocess, time, urllib.request, zipfile
from pathlib import Path
from google.colab import userdata
BIN, CONF, STORE = Path('/content/rclone/rclone'), Path('/content/rclone/rclone.conf'), Path('/content/bench-store')
EXPECTED = {'code/benchmark.py': 'fd8d34a6496ffa14ab58565e372c8a7f8dd350042a419b263685cf2149494ac2', 'code/supervisor.py': '81fbaf4df576dc8fc3046dcc3ad9dfd1b31802cee425879b35dc592542509c0b', 'inputs/input-lock.json': '0ceb4b72fd40fe810c77eaa517e52bf603254404bcb6be5c468e14b691abcc9b'}
if not BIN.exists():
    archive = urllib.request.urlopen('https://downloads.rclone.org/v1.75.1/rclone-v1.75.1-linux-amd64.zip', timeout=300).read()
    if hashlib.sha256(archive).hexdigest() != '982b5aa772841168f8e380f139e9e787b2a105403e32b94da8676a0e1c0a13ab':
        raise RuntimeError('rclone archive checksum mismatch')
    BIN.parent.mkdir(parents=True, exist_ok=True)
    BIN.write_bytes(zipfile.ZipFile(io.BytesIO(archive)).read('rclone-v1.75.1-linux-amd64/rclone')); BIN.chmod(0o700)
conf = userdata.get('TRUESKATE_BENCH_RCLONE')
if 'scope = drive.file' not in conf.splitlines() or not any(l.startswith('root_folder_id = ') for l in conf.splitlines()):
    raise RuntimeError('refusing a credential broader than one drive.file folder')
CONF.parent.mkdir(parents=True, exist_ok=True); CONF.write_text(conf); CONF.chmod(0o600)
os.environ.update(RCLONE_CONFIG=str(CONF), RCLONE_BIN=str(BIN))
for rel, digest in EXPECTED.items():
    subprocess.run([str(BIN), 'copyto', 'bench:' + rel, str(STORE / rel)], check=True)
    if hashlib.sha256((STORE / rel).read_bytes()).hexdigest() != digest:
        raise RuntimeError(rel + ' does not match this notebook')
spec = importlib.util.spec_from_file_location('supervisor', STORE / 'code/supervisor.py')
s = importlib.util.module_from_spec(spec); spec.loader.exec_module(s)
SESSION_ID = 'session-2'
s.confirm_release(SESSION_ID, time.time())


## Only if a GPU connection failed: on CPU, mark the prepared attempt failed (then use step 3's next_attempt within the cap).

In [ ]:
# Bootstrap: pinned rclone, drive.file-scoped credential, harness verified against this notebook.
import hashlib, importlib.util, io, os, subprocess, time, urllib.request, zipfile
from pathlib import Path
from google.colab import userdata
BIN, CONF, STORE = Path('/content/rclone/rclone'), Path('/content/rclone/rclone.conf'), Path('/content/bench-store')
EXPECTED = {'code/benchmark.py': 'fd8d34a6496ffa14ab58565e372c8a7f8dd350042a419b263685cf2149494ac2', 'code/supervisor.py': '81fbaf4df576dc8fc3046dcc3ad9dfd1b31802cee425879b35dc592542509c0b', 'inputs/input-lock.json': '0ceb4b72fd40fe810c77eaa517e52bf603254404bcb6be5c468e14b691abcc9b'}
if not BIN.exists():
    archive = urllib.request.urlopen('https://downloads.rclone.org/v1.75.1/rclone-v1.75.1-linux-amd64.zip', timeout=300).read()
    if hashlib.sha256(archive).hexdigest() != '982b5aa772841168f8e380f139e9e787b2a105403e32b94da8676a0e1c0a13ab':
        raise RuntimeError('rclone archive checksum mismatch')
    BIN.parent.mkdir(parents=True, exist_ok=True)
    BIN.write_bytes(zipfile.ZipFile(io.BytesIO(archive)).read('rclone-v1.75.1-linux-amd64/rclone')); BIN.chmod(0o700)
conf = userdata.get('TRUESKATE_BENCH_RCLONE')
if 'scope = drive.file' not in conf.splitlines() or not any(l.startswith('root_folder_id = ') for l in conf.splitlines()):
    raise RuntimeError('refusing a credential broader than one drive.file folder')
CONF.parent.mkdir(parents=True, exist_ok=True); CONF.write_text(conf); CONF.chmod(0o600)
os.environ.update(RCLONE_CONFIG=str(CONF), RCLONE_BIN=str(BIN))
for rel, digest in EXPECTED.items():
    subprocess.run([str(BIN), 'copyto', 'bench:' + rel, str(STORE / rel)], check=True)
    if hashlib.sha256((STORE / rel).read_bytes()).hexdigest() != digest:
        raise RuntimeError(rel + ' does not match this notebook')
spec = importlib.util.spec_from_file_location('supervisor', STORE / 'code/supervisor.py')
s = importlib.util.module_from_spec(spec); spec.loader.exec_module(s)
s.record_failed_connection('describe the observed failure')


## At the end, on CPU: record active operator time; unknown effort stays unknown.

In [ ]:
# Bootstrap: pinned rclone, drive.file-scoped credential, harness verified against this notebook.
import hashlib, importlib.util, io, os, subprocess, time, urllib.request, zipfile
from pathlib import Path
from google.colab import userdata
BIN, CONF, STORE = Path('/content/rclone/rclone'), Path('/content/rclone/rclone.conf'), Path('/content/bench-store')
EXPECTED = {'code/benchmark.py': 'fd8d34a6496ffa14ab58565e372c8a7f8dd350042a419b263685cf2149494ac2', 'code/supervisor.py': '81fbaf4df576dc8fc3046dcc3ad9dfd1b31802cee425879b35dc592542509c0b', 'inputs/input-lock.json': '0ceb4b72fd40fe810c77eaa517e52bf603254404bcb6be5c468e14b691abcc9b'}
if not BIN.exists():
    archive = urllib.request.urlopen('https://downloads.rclone.org/v1.75.1/rclone-v1.75.1-linux-amd64.zip', timeout=300).read()
    if hashlib.sha256(archive).hexdigest() != '982b5aa772841168f8e380f139e9e787b2a105403e32b94da8676a0e1c0a13ab':
        raise RuntimeError('rclone archive checksum mismatch')
    BIN.parent.mkdir(parents=True, exist_ok=True)
    BIN.write_bytes(zipfile.ZipFile(io.BytesIO(archive)).read('rclone-v1.75.1-linux-amd64/rclone')); BIN.chmod(0o700)
conf = userdata.get('TRUESKATE_BENCH_RCLONE')
if 'scope = drive.file' not in conf.splitlines() or not any(l.startswith('root_folder_id = ') for l in conf.splitlines()):
    raise RuntimeError('refusing a credential broader than one drive.file folder')
CONF.parent.mkdir(parents=True, exist_ok=True); CONF.write_text(conf); CONF.chmod(0o600)
os.environ.update(RCLONE_CONFIG=str(CONF), RCLONE_BIN=str(BIN))
for rel, digest in EXPECTED.items():
    subprocess.run([str(BIN), 'copyto', 'bench:' + rel, str(STORE / rel)], check=True)
    if hashlib.sha256((STORE / rel).read_bytes()).hexdigest() != digest:
        raise RuntimeError(rel + ' does not match this notebook')
spec = importlib.util.spec_from_file_location('supervisor', STORE / 'code/supervisor.py')
s = importlib.util.module_from_spec(spec); spec.loader.exec_module(s)
s.record_manual_effort([{'what': 'describe', 'seconds': 0}], complete=False)
